In [1]:
# notebook : semdetect_e3.ipynb — cellule 0 : imports et configuration
import json
import logging
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd

from src.data.dataset_aprdb import AprdbDataset
from src.detector.sem_detect.semdetect.review_cleaner import strip_markdown

logging.basicConfig(level=logging.INFO)
logging.getLogger("httpx").setLevel(logging.WARNING)

CACHE_ROOT = Path(os.environ["CACHE_DIR"])
# configuration identique à l'évaluation SAGE (sage_gru_2_cat) -> mêmes cibles, même cache
APRDB_BASE = {
    "name": "aprdb",
    "description": "AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt",
    "conference": None, "date": None, "keep_meta": True,
    "split": "test", "with_pdf": False, "seed": 0,
}
SECONDS_PER_EXTRACTION = 1.9   # Qwen3-32B, mesuré en E1 (32 requêtes en parallèle, GPU 1)

/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-05 03:27:28.897 | INFO     | src.config:<module>:12 - PROJ_ROOT path is: /home/cyaiche/generated_text_detection_hub


/data2/cyaiche/cache_dir


In [2]:
# notebook : semdetect_e3.ipynb — cellule 1 : chargement et unicité des identifiants
pool = AprdbDataset(config={**APRDB_BASE, "subset_size": None}).data.to_pandas()     # réservoir de références
evald = AprdbDataset(config={**APRDB_BASE, "subset_size": 10000}).data.to_pandas()   # cibles évaluées par SAGE
for d in (pool, evald):
    d["class"] = np.where(d["label"] == 1, "ai", "human")
    # identifiant unique par ligne : le review-id est partagé entre l'humaine et ses jumelles IA
    d["uid"] = d["review-id"].astype(str) + "::" + np.where(d["label"] == 1, d["generator"], "human")

print(f"réservoir : {len(pool)} reviews | cibles SAGE : {len(evald)}")
print(f"années : {sorted(pool['year'].unique())} | conférences : {sorted(pool['conference'].unique())}\n")
print(pd.crosstab(pool["generator"], pool["class"]), "\n")

print("lignes humaines par review-id :", pool[pool["class"] == "human"].groupby("review-id").size().value_counts().to_dict())
print("lignes IA par (review-id, générateur) :",
      pool[pool["class"] == "ai"].groupby(["review-id", "generator"]).size().value_counts().to_dict())
print(f"uid unique dans le réservoir : {pool['uid'].is_unique} | dans les cibles : {evald['uid'].is_unique}")
print(f"cibles présentes dans le réservoir : {evald['uid'].isin(pool['uid']).mean():.1%}")

Loading APRDB dataset from IntelLabs/AI-Peer-Review-Detection-Benchmark with config: {'name': 'aprdb', 'description': 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt', 'conference': None, 'date': None, 'keep_meta': True, 'split': 'test', 'with_pdf': False, 'seed': 0, 'subset_size': None}
📦 Loading dataset from cache: /data2/cyaiche/cache_dir/dataset_cache/dataset_b2c9f76fec09d101fd80daf552062d9b.pt
Loading APRDB dataset from IntelLabs/AI-Peer-Review-Detection-Benchmark with config: {'name': 'aprdb', 'description': 'AI Peer Review Detection Benchmark, test subset, binaire mgt/hwt', 'conference': None, 'date': None, 'keep_meta': True, 'split': 'test', 'with_pdf': False, 'seed': 0, 'subset_size': 10000}
📦 Loading dataset from cache: /data2/cyaiche/cache_dir/dataset_cache/dataset_13af569ff61302c3c2d7fbb1eea34954.pt
réservoir : 101890 reviews | cibles SAGE : 9468
années : ['2016', '2017', '2018', '2019', '2020', '2021'] | conférences : ['ICLR', 'NeurIPS']

class         ai

In [3]:
# notebook : semdetect_e3.ipynb — cellule 2 : couverture des références natives (R2)
mgt = pool[pool["class"] == "ai"]
gens = mgt.groupby("review-id")["generator"].agg(set)
print("générateurs IA distincts par review-id :", gens.map(len).value_counts().sort_index().to_dict(), "\n")


def n_refs_available(row) -> int:
    g = gens.get(row["review-id"], set())
    return len(g) if row["class"] == "human" else len(g - {row["generator"]})


evald["n_refs"] = evald.apply(n_refs_available, axis=1)
evald["refs_ok"] = evald["n_refs"] >= 3
print("cibles avec au moins 3 références disponibles :")
print(evald.groupby("class")["refs_ok"].agg(["sum", "count", "mean"]).round(3), "\n")
print("cibles IA, par générateur :")
print(evald[evald["class"] == "ai"].groupby("generator")["refs_ok"].agg(["sum", "count", "mean"]).round(3))

générateurs IA distincts par review-id : {3: 3, 4: 187, 5: 16823} 

cibles avec au moins 3 références disponibles :
        sum  count  mean
class                   
ai     4999   5000   1.0
human  4468   4468   1.0 

cibles IA, par générateur :
            sum  count   mean
generator                    
claude      990    990  1.000
gemini      980    980  1.000
gpt4o       993    994  0.999
llama      1000   1000  1.000
qwen       1036   1036  1.000


In [4]:
# notebook : semdetect_e3.ipynb — cellule 3 : format et longueur, avant et après strip_markdown
PATTERNS = {
    "titres #": r"(?m)^\s*#+\s",
    "gras **": r"\*\*",
    "puces - * •": r"(?m)^\s*[-*•]\s+",
    "listes numérotées": r"(?m)^\s*\d+[.)]\s+",
    "préambule LLM": r"(?i)^\s*(here is|here's|sure|certainly)\b",
    "\\n littéral": r"\\n",
    "fragment JSON": r'"\s*(summary|main_review|review|strengths|weaknesses)\w*"\s*:',
}


def marker_table(texts: pd.Series, classes: pd.Series) -> pd.DataFrame:
    m = pd.DataFrame({k: texts.str.contains(p, regex=True) for k, p in PATTERNS.items()})
    return m.groupby(classes.to_numpy()).mean().T


cleaned = pool["content"].map(strip_markdown)
print("Part des reviews contenant chaque marqueur (brut | après strip_markdown) :")
print(pd.concat({"brut": marker_table(pool["content"], pool["class"]),
                 "nettoyé": marker_table(cleaned, pool["class"])}, axis=1).round(3), "\n")

pool["n_chars"] = cleaned.str.len()
print("longueur médiane (caractères, après nettoyage) par conférence, année et classe :")
print(pool.pivot_table(index=["conference", "year"], columns="class", values="n_chars", aggfunc="median").round(0), "\n")
print("part des reviews de moins de 800 caractères (risque < 4 claims sémantiques) :")
print((pool["n_chars"] < 800).groupby(pool["class"]).mean().round(3).to_dict())

Part des reviews contenant chaque marqueur (brut | après strip_markdown) :


/tmp/ipykernel_1184398/3022412848.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m = pd.DataFrame({k: texts.str.contains(p, regex=True) for k, p in PATTERNS.items()})
/tmp/ipykernel_1184398/3022412848.py:14: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  m = pd.DataFrame({k: texts.str.contains(p, regex=True) for k, p in PATTERNS.items()})


                    brut        nettoyé       
                      ai  human      ai  human
titres #           0.125  0.036   0.000  0.000
gras **            0.374  0.041   0.000  0.000
puces - * •        0.696  0.224   0.507  0.189
listes numérotées  0.280  0.158   0.280  0.158
préambule LLM      0.000  0.000   0.000  0.000
\n littéral        0.001  0.011   0.001  0.011
fragment JSON      0.000  0.000   0.000  0.000 

longueur médiane (caractères, après nettoyage) par conférence, année et classe :
class                ai   human
conference year                
ICLR       2017  2479.0  1626.0
           2018  2507.0  1996.0
           2019  2527.0  2098.0
           2020  2961.0  2133.0
           2021  3320.0  2564.0
NeurIPS    2016  2456.0  1784.0
           2017  2849.0  1774.0
           2018  3022.0  2169.0
           2019  2654.0  1800.0
           2021  1664.0  1929.0 

part des reviews de moins de 800 caractères (risque < 4 claims sémantiques) :
{'ai': 0.005, 'human': 0.085}


In [5]:
# notebook : semdetect_e3.ipynb — cellule 4 : volume d'extraction des claims
touched = set(evald["review-id"])
mgt_touched = mgt[mgt["review-id"].isin(touched)]
extra_refs = mgt_touched[~mgt_touched["uid"].isin(set(evald["uid"]))]
n_full = len(evald) + len(extra_refs)
print(f"cibles : {len(evald)} | review-id touchés : {len(touched)}")
print(f"jumelles IA hors cibles (références potentielles, borne haute) : {len(extra_refs)}")
print(f"=> extractions, évaluation complète : {n_full} ≈ {n_full * SECONDS_PER_EXTRACTION / 3600:.1f} h")

# pilote : un tirage de review-id, toutes leurs cibles et leurs jumelles
rng = np.random.default_rng(0)
for n_ids in (500, 1000, 2000):
    ids = set(rng.choice(sorted(touched), size=min(n_ids, len(touched)), replace=False))
    n_t = int(evald["review-id"].isin(ids).sum())
    n_r = int(extra_refs["review-id"].isin(ids).sum())
    print(f"pilote {n_ids} review-id : {n_t} cibles + {n_r} références = {n_t + n_r} "
          f"≈ {(n_t + n_r) * SECONDS_PER_EXTRACTION / 3600:.1f} h")

cibles : 9468 | review-id touchés : 4468
jumelles IA hors cibles (références potentielles, borne haute) : 17304
=> extractions, évaluation complète : 26772 ≈ 14.1 h
pilote 500 review-id : 1058 cibles + 1937 références = 2995 ≈ 1.6 h
pilote 1000 review-id : 2119 cibles + 3874 références = 5993 ≈ 3.2 h
pilote 2000 review-id : 4226 cibles + 7759 références = 11985 ≈ 6.3 h


In [6]:
# notebook : semdetect_e3.ipynb — cellule 5 : exclusions, pilote et univers de calcul
from concurrent.futures import ThreadPoolExecutor

from src.detector.sem_detect.text_cleaning import CLEANERS

E3_DIR = CACHE_ROOT / "sem_detect_e3"
E3_DIR.mkdir(parents=True, exist_ok=True)
N_PILOT_IDS = 1000

# 1) couples (review-id, générateur) ambigus : les deux lignes sont exclues
is_ai = pool["class"].eq("ai")
dup = is_ai & pool.duplicated(["review-id", "generator"], keep=False)
ambiguous = set(pool.loc[dup, "uid"])
pool_ok = pool[~pool["uid"].isin(ambiguous)].copy()
evald_ok = evald[~evald["uid"].isin(ambiguous)].copy()
print(f"exclus : {int(dup.sum())} lignes du réservoir, {len(evald) - len(evald_ok)} cibles")

# 2) pilote : tirage déterministe de review-id parmi ceux des cibles SAGE
for d in (pool_ok, evald_ok):
    d["group"] = d["review-id"].astype(str)
rng = np.random.default_rng(0)
pilot_ids = set(rng.choice(sorted(evald_ok["group"].unique()), size=N_PILOT_IDS, replace=False))

targets = evald_ok[evald_ok["group"].isin(pilot_ids)]
refs_pool = pool_ok[pool_ok["group"].isin(pilot_ids) & pool_ok["class"].eq("ai")]
universe = pd.concat([targets, refs_pool], ignore_index=True).drop_duplicates("uid")
universe["is_target"] = universe["uid"].isin(set(targets["uid"]))
universe = universe[["uid", "group", "class", "generator", "conference", "year", "is_target", "content"]]
universe.to_parquet(E3_DIR / "pilot_universe.parquet", index=False)

print(f"pilote : {len(pilot_ids)} review-id | {universe['is_target'].sum()} cibles "
      f"({targets['class'].value_counts().to_dict()}) | {len(universe)} reviews à extraire")


def variant_frame(variant: str) -> pd.DataFrame:
    """Univers du pilote pour une variante de nettoyage : identifiants préfixés, textes nettoyés."""
    v = universe.copy()
    v["review-id"] = f"aprdb:{variant}:" + v["uid"]
    v["content"] = v["content"].map(CLEANERS[variant])
    return v

exclus : 17022 lignes du réservoir, 990 cibles
pilote : 1000 review-id | 1918 cibles ({'human': 1000, 'ai': 918}) | 4996 reviews à extraire


In [7]:
# notebook : semdetect_e3.ipynb — cellule 6 : claims (GPU 1) et features textuelles (GPU 0) en parallèle
from src.detector.sem_detect.claims_llm import LocalClaimExtractor
from src.detector.sem_detect.feature_store import SemDetectFeatureStore
from src.utils.vllm_manager import VLLMManager

STORE_ROOT = CACHE_ROOT / "sem_detect_store"
FEATURES_DEVICE = "cuda:0"

mgr = VLLMManager(gpu=1, restart_watchdog=True)
mgr.ensure("qwen3-32b-batch")
extractor = LocalClaimExtractor(cache_root=CACHE_ROOT, served_model="qwen3-32b", max_workers=32)

frames, stores, claims = {}, {}, {}
for variant in ("a", "b"):
    df_v = variant_frame(variant)
    store = SemDetectFeatureStore(root=STORE_ROOT, claims_source=extractor.source_tag,
                                  text_source=f"aprdb:test:clean_{variant}", device=FEATURES_DEVICE)
    t0 = time.perf_counter()
    with ThreadPoolExecutor(max_workers=2) as ex:
        fut_claims = ex.submit(extractor.extract, df_v[["review-id", "content"]])
        fut_text = ex.submit(store.ensure_textual, df_v.loc[df_v["is_target"], ["review-id", "content"]])
        out = fut_claims.result()
        fut_text.result()
    out = extractor.retry_failed(df_v[["review-id", "content"]], max_tokens=8192)
    out = extractor.retry_failed(df_v[["review-id", "content"]], max_tokens=8192,
                                 extra_sampling={"repetition_penalty": 1.05})
    store.ensure_embeddings(df_v[["review-id"]].merge(out[["review-id", "claim_extraction"]], on="review-id"))

    frames[variant], stores[variant], claims[variant] = df_v, store, out
    print(f"\nvariante {variant} : {(time.perf_counter() - t0) / 60:.0f} min | claims "
          f"{out['claims_status'].value_counts().to_dict()} | stats textuelles {store.n_textual()}")

INFO:src.utils.vllm_manager:qwen3-32b déjà servi et opérationnel : réutilisé
INFO:src.detector.sem_detect.claims_llm:Claims cache /data2/cyaiche/cache_dir/claims_llm/vllm_qwen3-32b_0902bb5d60.jsonl : 5430 ok
INFO:src.detector.sem_detect.feature_store:Feature store : 5430 embeddings (fbbbe1608d0e), 0 stats textuelles (cdedf1ab8567)
Textuel fragment 1/8:   0%|          | 0/256 [00:00<?, ?it/s]INFO:src.detector.sem_detect.semdetect.textual_features:Textual features device: cuda:0
INFO:src.detector.sem_detect.semdetect.textual_features:Loading textual features model: mistralai/Mistral-7B-Instruct-v0.3

Claims (qwen3-32b):   0%|          | 0/4996 [00:00<?, ?it/s][transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Claims (qwen3-32b):   0%|          | 2/4996 [00:23<13:59:24, 10.08s/it]

Loading weights:   0%|          | 1/291 [00:21<1:41:45, 21.05s/it]

Claims (qwen3-32b):   0%|          | 3/4996 [00:29<11:16:00,  8.12s/it]

Claims (qwen3-32b):   0%|          | 4/4996 [00:34<9:


variante a : 109 min | claims {'ok': 4996} | stats textuelles 1918


INFO:src.detector.sem_detect.feature_store:Feature store : 10426 embeddings (fbbbe1608d0e), 0 stats textuelles (cf890d2baa4a)
Textuel fragment 1/8:   0%|          | 0/256 [00:00<?, ?it/s]INFO:src.detector.sem_detect.semdetect.textual_features:Textual features device: cuda:0
INFO:src.detector.sem_detect.semdetect.textual_features:Loading textual features model: mistralai/Mistral-7B-Instruct-v0.3

Claims (qwen3-32b):   0%|          | 0/4996 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Loading weights:  26%|██▌       | 76/291 [00:00<00:00, 732.18it/s]

Loading weights:  52%|█████▏    | 150/291 [00:00<00:00, 619.19it/s]

Loading weights:  76%|███████▌  | 220/291 [00:00<00:00, 628.13it/s]

Loading weights: 100%|██████████| 291/291 [00:00<00:00, 560.05it/s]

Textuel fragment 8/8: 100%|██████████| 126/126 [00:25<00:00,  4.91it/s]]

Claims (qwen3-32b): 100%|██████████| 4996/4996 [1:49:28<00:00,  1.31s/it]
INFO:src.detector.sem_detect.claims_llm:relance de 4 échec


variante b : 119 min | claims {'ok': 4996} | stats textuelles 1918


In [8]:
# notebook : semdetect_e3.ipynb — cellule 7 : évaluation du pilote, variantes (a) et (b)
from sklearn.metrics import roc_auc_score

from src.config import PROJ_ROOT
from src.detector.sem_detect.references import build_feature_matrix, paper_protocol_references, predict_proba
from src.detector.sem_detect.semdetect.classifier import FEATURE_DISPLAY_NAMES, SemDetectClassifier
from src.evaluation.metrics import get_tpr_at_fpr

clf = SemDetectClassifier(str(PROJ_ROOT / "src/detector/sem_detect/models/lightgbm_best_model.pkl"))
PRED_NAMES = ["ai", "rewrite", "human"]


def evaluate_variant(variant: str):
    df_v, store = frames[variant], stores[variant]
    refs = paper_protocol_references(df_v, k=3, seed=0, key_cols=["group"],
                                     is_valid=lambda rid: store.semantic(rid, 1) is not None)
    tids = df_v.loc[df_v["is_target"], "review-id"].tolist()
    X, kept, skipped = build_feature_matrix(store, tids, refs, clf.similarity_threshold)
    P = pd.DataFrame(predict_proba(clf, X), index=kept, columns=["p_ai", "p_rewrite", "p_human"])
    meta = df_v.set_index("review-id").loc[kept, ["class", "generator", "conference", "year"]]
    return P, pd.DataFrame(X, index=kept, columns=FEATURE_DISPLAY_NAMES), meta, skipped


def binary_summary(P: pd.DataFrame, meta: pd.DataFrame) -> dict:
    y, s = meta["class"].eq("ai").to_numpy(), P["p_ai"].to_numpy()
    pred = np.array(PRED_NAMES)[P.to_numpy().argmax(axis=1)]
    return {
        "n classées": len(y), "AUC": roc_auc_score(y, s),
        "TPR@1%": get_tpr_at_fpr(y, s, 0.01), "TPR@5%": get_tpr_at_fpr(y, s, 0.05),
        "IA -> ai": (pred[y] == "ai").mean(), "IA -> rewrite": (pred[y] == "rewrite").mean(),
        "IA -> human": (pred[y] == "human").mean(),
        "humain -> ai (FPR argmax)": (pred[~y] == "ai").mean(), "humain -> rewrite": (pred[~y] == "rewrite").mean(),
    }


evals = {v: evaluate_variant(v) for v in ("a", "b")}

print("=== Pilote APRDB : humain vs IA (score P(ai)) et décisions à 3 classes ===")
print(pd.DataFrame({f"({v})": binary_summary(P, meta) for v, (P, _, meta, _) in evals.items()}).round(4), "\n")

print("=== Cibles non classables ===")
for v, (_, _, _, skipped) in evals.items():
    s = pd.Series(skipped, dtype=object)
    cls = frames[v].set_index("review-id")["class"]
    print(f"({v}) : {len(s)} {s.to_frame('raison').join(cls).value_counts().to_dict() if len(s) else {}}")
print()

for v, (P, _, meta, _) in evals.items():
    y, s = meta["class"].eq("ai"), P["p_ai"]
    humans = s[~y]
    rows = []
    for gen, idx in meta[y].groupby("generator").groups.items():
        part = pd.concat([s.loc[idx], humans])
        lab = np.r_[np.ones(len(idx)), np.zeros(len(humans))]
        rows.append({"générateur": gen, "n": len(idx), "AUC vs humains": roc_auc_score(lab, part),
                     "TPR@1%": get_tpr_at_fpr(lab, part.to_numpy(), 0.01),
                     "rappel argmax": (P.loc[idx].to_numpy().argmax(axis=1) == 0).mean()})
    print(f"=== ({v}) par générateur de la cible ===\n{pd.DataFrame(rows).set_index('générateur').round(3)}\n")

    rows = []
    for (conf, year), g in meta.groupby(["conference", "year"]):
        if g["class"].nunique() == 2:
            rows.append({"conférence": conf, "année": year, "n": len(g),
                         "AUC": roc_auc_score(g["class"].eq("ai"), s.loc[g.index])})
    print(f"=== ({v}) par conférence et année ===\n{pd.DataFrame(rows).set_index(['conférence', 'année']).round(3)}\n")

print("=== Moyenne des 9 features par classe (référence : moyenne du train) ===")
table = {"train": pd.Series(clf.scaler.mean_, index=FEATURE_DISPLAY_NAMES)}
for v, (_, X, meta, _) in evals.items():
    for c in ("ai", "human"):
        table[f"({v}) {c}"] = X[meta["class"].eq(c)].mean()
print(pd.DataFrame(table).round(4))

INFO:src.detector.sem_detect.semdetect.classifier:Loading Sem-Detect model from /home/cyaiche/generated_text_detection_hub/src/detector/sem_detect/models/lightgbm_best_model.pkl
/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/cyaiche/generated_text_detection_hub/.venv/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.8.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintai

=== Pilote APRDB : humain vs IA (score P(ai)) et décisions à 3 classes ===
                                 (a)        (b)
n classées                 1850.0000  1850.0000
AUC                           0.9799     0.9821
TPR@1%                        0.6994     0.6482
TPR@5%                        0.9227     0.9281
IA -> ai                      0.4210     0.4204
IA -> rewrite                 0.5713     0.5686
IA -> human                   0.0077     0.0111
humain -> ai (FPR argmax)     0.0021     0.0011
humain -> rewrite             0.2582     0.2368 

=== Cibles non classables ===
(a) : 68 {('cible < 4 claims sémantiques', 'human'): 55, ('cible < 4 claims sémantiques', 'ai'): 8, ('références insuffisantes (2/3)', 'ai'): 5}
(b) : 68 {('cible < 4 claims sémantiques', 'human'): 54, ('cible < 4 claims sémantiques', 'ai'): 9, ('références insuffisantes (2/3)', 'ai'): 5}

=== (a) par générateur de la cible ===
              n  AUC vs humains  TPR@1%  rappel argmax
générateur                  